In [1]:
# THis took around 30 minutes for two sessions

In [1]:
import sys
from pathlib import Path

import UnitMatchPy.Extract_raw_data as erd
import numpy as np 
from pathlib import Path
from joblib import Parallel, delayed
import matplotlib.pyplot as plt
import os

In [2]:
#Set Up Parameters
SampleAmount = 1000 # for both CV, at least 500 per CV
SpikeWidth = 82 # assuming 30khz sampling, 82 and 61 are common choices, covers the AP and space around needed for processing
HalfWidth = np.floor(SpikeWidth/2).astype(int)
nChannels = 383 #neuropixels default 384 is default but do i need to remove the reference channel?
ExtractGoodUnitsOnly = True # bool, set to true if you want to only extract units marked as good 

KS4data = False #bool, set to true if using Kilosort, as KS4 spiketimes refer to start of waveform not peak
if KS4data:
    SamplesBefore = 20
    SamplesAfter = SpikeWidth - SamplesBefore

#List of paths to a KS directory, can pass paths 
KSdirs = [
    r"W:\branco\Laurence\JAL006\JAL006_shelter_barrier_flip_3_2024_03_21T11_20_34\barrier_flip3_g0\barrier_flip3_g0_imec0\SI_KS_output\sorter_output",
    r"W:\branco\Laurence\JAL006\JAL006_shelter_barrier_flip_5_2024_03_25T11_05_33\barrier_flip5_g0\barrier_flip5_g0_imec0\SI_KS_output\sorter_output"
]
nSessions = len(KSdirs) #How many session are being extracted
SpikeIds, SpikeTimes, GoodUnits = erd.extract_KSdata(KSdirs, ExtractGoodUnitsOnly = True)
print(SpikeIds)

[array([[ 49],
       [111],
       [141],
       ...,
       [220],
       [ 77],
       [ 65]], dtype=uint32), array([307, 135, 331, ..., 284, 275, 312])]


In [3]:
#give metadata + Raw data paths
#if you are NOT decompressing data here, provide a list of paths to the decompressed data and the metadata

DataPaths = [
    r"W:\branco\Laurence\JAL006\JAL006_shelter_barrier_flip_3_2024_03_21T11_20_34\barrier_flip3_g0\barrier_flip3_g0_imec0\barrier_flip3_g0_t0.imec0.ap.bin",
    r"W:\branco\Laurence\JAL006\JAL006_shelter_barrier_flip_5_2024_03_25T11_05_33\barrier_flip5_g0\barrier_flip5_g0_imec0\barrier_flip5_g0_t0.imec0.ap.bin"
]
    
metaPaths = [
    r"W:\branco\Laurence\JAL006\JAL006_shelter_barrier_flip_3_2024_03_21T11_20_34\barrier_flip3_g0\barrier_flip3_g0_imec0\barrier_flip3_g0_t0.imec0.ap.meta",
    r"W:\branco\Laurence\JAL006\JAL006_shelter_barrier_flip_5_2024_03_25T11_05_33\barrier_flip5_g0\barrier_flip5_g0_imec0\barrier_flip5_g0_t0.imec0.ap.meta"
]

In [4]:
#Extract the units 
if ExtractGoodUnitsOnly:
    print('Extracting Good Units Only')
    for sid in range(nSessions):
        #load metadata
        MetaData = erd.Read_Meta(Path(metaPaths[sid]))
        nElements = int(MetaData['fileSizeBytes']) / 2
        nChannelsTot = int(MetaData['nSavedChans'])

        #create memmap to raw data, for that session
        Data = np.memmap(DataPaths[sid], dtype = 'int16', shape =(int(nElements / nChannelsTot), nChannelsTot))

        # Remove spike which won't have a full wavefunction recorded
        # Ensure the boolean array is one-dimensional and matches the length of SpikeTimes[sid]
        not_valid_spikes = np.logical_or((SpikeTimes[sid] < HalfWidth), (SpikeTimes[sid] > (Data.shape[0] - HalfWidth)))
        SpikeIdsTmp = np.delete(SpikeIds[sid], np.where(not_valid_spikes))
        SpikeTimesTmp = np.delete(SpikeTimes[sid], np.where(not_valid_spikes))
        
        #might be slow extracting smaple for good units only?
        SampleIdx = erd.get_sample_idx(SpikeTimesTmp, SpikeIdsTmp, SampleAmount, units = GoodUnits[sid])
        

        if KS4data:
            AvgWaveforms = Parallel(n_jobs = -1, verbose = 10, mmap_mode='r', max_nbytes=None )(delayed(erd.Extract_A_UnitKS4)(SampleIdx[uid], Data, SamplesBefore, SamplesAfter, SpikeWidth, nChannels, SampleAmount)for uid in range(GoodUnits[sid].shape[0]))
            AvgWaveforms = np.asarray(AvgWaveforms)           
        else:
            AvgWaveforms = Parallel(n_jobs = -1, verbose = 10, mmap_mode='r', max_nbytes=None )(delayed(erd.Extract_A_Unit)(SampleIdx[uid], Data, HalfWidth, SpikeWidth, nChannels, SampleAmount)for uid in range(GoodUnits[sid].shape[0]))
            AvgWaveforms = np.asarray(AvgWaveforms)

        #Save in file named 'RawWaveforms' in the KS Directory
        erd.Save_AvgWaveforms(AvgWaveforms, KSdirs[sid], GoodUnits = GoodUnits[sid], ExtractGoodUnitsOnly = ExtractGoodUnitsOnly)

else:
    for sid in range(nSessions):
        #Extracting ALL the Units
        nUnits = len(np.unique(SpikeIds[sid]))
        #load metadata
        MetaData = erd.Read_Meta(Path(metaPaths[sid]))
        nElements = int(MetaData['fileSizeBytes']) / 2
        nChannelsTot = int(MetaData['nSavedChans'])

        #create memmap to raw data, for that session
        Data = np.memmap(DataPaths[sid], dtype = 'int16', shape =(int(nElements / nChannelsTot), nChannelsTot))

        # Remove spike which won't have a full wavefunction recorded
        SpikeIdsTmp = np.delete(SpikeIds[sid], np.logical_or( (SpikeTimes[sid] < HalfWidth), ( SpikeTimes[sid] > (Data.shape[0] - HalfWidth))))
        SpikeTimesTmp = np.delete(SpikeTimes[sid], np.logical_or( (SpikeTimes[sid] < HalfWidth), ( SpikeTimes[sid] > (Data.shape[0] - HalfWidth))))


        SampleIdx = erd.get_sample_idx(SpikeTimesTmp, SpikeIdsTmp, SampleAmount, units= np.unique(SpikeIds[sid]))
        
        if KS4data:
            AvgWaveforms = Parallel(n_jobs = -1, verbose = 10, mmap_mode='r', max_nbytes=None )(delayed(erd.Extract_A_UnitKS4)(SampleIdx[uid], Data, SamplesBefore, SamplesAfter, SpikeWidth, nChannels, SampleAmount)for uid in range(nUnits))
            AvgWaveforms = np.asarray(AvgWaveforms)           
        else:
            AvgWaveforms = Parallel(n_jobs = -1, verbose = 10, mmap_mode='r', max_nbytes=None )(delayed(erd.Extract_A_Unit)(SampleIdx[uid], Data, HalfWidth, SpikeWidth, nChannels, SampleAmount)for uid in range(nUnits))
            AvgWaveforms = np.asarray(AvgWaveforms)

        #Save in file named 'RawWaveforms' in the KS Directory
        erd.Save_AvgWaveforms(AvgWaveforms, KSdirs[sid], GoodUnits = GoodUnits[sid], ExtractGoodUnitsOnly = ExtractGoodUnitsOnly)
del Data 

Extracting Good Units Only


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 32 concurrent workers.
[Parallel(n_jobs=-1)]: Done   8 tasks      | elapsed:   28.1s
[Parallel(n_jobs=-1)]: Done  21 tasks      | elapsed:   28.3s
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:   54.6s
[Parallel(n_jobs=-1)]: Done  49 tasks      | elapsed:   54.9s
[Parallel(n_jobs=-1)]: Done  64 tasks      | elapsed:   55.0s
[Parallel(n_jobs=-1)]: Done  81 tasks      | elapsed:  1.4min
[Parallel(n_jobs=-1)]: Done  98 tasks      | elapsed:  1.5min
[Parallel(n_jobs=-1)]: Done 117 tasks      | elapsed:  1.9min
[Parallel(n_jobs=-1)]: Done 136 tasks      | elapsed:  2.3min
[Parallel(n_jobs=-1)]: Done 157 tasks      | elapsed:  2.4min
[Parallel(n_jobs=-1)]: Done 178 tasks      | elapsed:  2.8min
[Parallel(n_jobs=-1)]: Done 201 tasks      | elapsed:  3.3min
[Parallel(n_jobs=-1)]: Done 241 out of 276 | elapsed:  3.8min remaining:   33.4s
[Parallel(n_jobs=-1)]: Done 269 out of 276 | elapsed:  4.1min remaining:    6.4s
[Parallel(n_jobs=